In [1]:
!pip install llama-index llama-parse

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 41.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.5/328.5 kB 26.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 15.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 22.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.3 MB/s eta 0:00:00


In [2]:
import nest_asyncio
nest_asyncio.apply()

In [42]:
import os
from google.colab import userdata
os.environ["LLAMA_CLOUD_API_KEY"]= 'llx-t9G9JztBDJXPUdksWOnkD2LSf4bQ9jQLwJIjNwDoHdFMmRST'
os.environ["OPENAI_API_KEY"]= 'sk-proj-0ha9cPU4MMNqPqHhL4WAT3BlbkFJkpkWoRSZzu0TXy2OiRr3'

In [43]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core import Settings


In [44]:
embed_model=OpenAIEmbedding(model="text-embedding-3-small")
llm = OpenAI(model="gpt-3.5-turbo-0125")
Settings.llm = llm

In [45]:
from llama_parse import LlamaParse

In [58]:
documents =  LlamaParse(result_type="markdown").load_data("Uttarakhand.pdf")

Started parsing the file under job_id 1b268a10-bca7-4b3d-9e87-3adfadd76820
......

In [59]:
print(documents[0].text)

Government of India

Ministry of Health and Family Welfare

Uttarakhand

National Family Health Survey (NFHS-5) 2019-21

India

International Institute for Population Sciences
Deonar, Mumbai 400 088


In [60]:
from llama_index.core.node_parser import MarkdownElementNodeParser
node_parser = MarkdownElementNodeParser(llm=OpenAI(model="gpt-3.5-turbo-0125"), num_workers=8)

In [61]:
nodes = node_parser.get_nodes_from_documents(documents)

0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
1it [00:00, 7543.71it/s]
  0%|          | 0/1 [00:00<?, ?it/s]WARNING:llama_index.core.response_synthesizers.refine:Validation error on structured response: 1 validation error for TableOutput
columns
  field required (type=value_error.missing)
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/llama_index/core/response_synthesizers/refine.py", line 482, in _agive_response_single
    structured_response = await program.acall(
  File "/usr/local/lib/python3.10/dist-packages/llama_index/core/response_synthesizers/refine.py", line 92, in acall
    answer = await self._llm.astructured_predict(
  File "/usr/local/lib/python3.10/dist-packages/llama_index/core/instrumentation/dispatcher.py", line 255, in async_wrapper
    result = await f

In [62]:
base_nodes, objects = node_parser.get_nodes_and_objects(nodes)
recursive_index =  VectorStoreIndex(nodes=base_nodes+objects)

In [63]:
query_engine= recursive_index.as_query_engine(similarity_top_k=25)

In [64]:
query1= "Percentage of women of age more than 12 years who own a mobile phone "
response1 =query_engine.query(query1)
print(str(response1))

20.1


In [4]:
import pickle
def load_elements(output_path_folder):
    with open(output_path_folder,'rb') as f:
        elements = pickle.load(f)
        return elements
all_elements = load_elements(output_path_folder='Pdffiles/processeddata.pkl')

In [7]:
category_counts = {}
for element in all_elements:
    category = str(type(element))
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1
unique_categories = set(category_counts.keys())
category_counts

{"<class 'unstructured.documents.elements.CompositeElement'>": 410,
 "<class 'unstructured.documents.elements.Table'>": 135,
 "<class 'unstructured.documents.elements.TableChunk'>": 57}